# CS781 Phase 6 — Abstractive legal summarisation with our Phase-1 models

Every Phase-1 model is used as a conditional generator p(summary | document): PrefixLM = document as the bidirectional prefix; T5 = document in the encoder; Llama / RetNet = document then summary in one causal sequence. **Long documents:** importance-based context selection (opening + ending + most central sentences) and, when references are long, **segment-wise summarisation**: the judgment is cut into up to 4 segments, each reference-summary sentence is aligned to the segment it overlaps most, one summary is generated per segment and the parts are concatenated (repeated sentences removed). Decoding: greedy with a KV cache, no repeated trigrams, minimum length. **Techniques:** LoRA, LoRMA, Houlsby and Pfeiffer adapters, prompt tuning, prefix tuning, word-level knowledge distillation (PrefixLM full FT -> 6-layer RetNet).

**Two sets** (set `RUN_SET` in the first cell). **Kaggle:** GPU T4, Internet on, secret `HF`/`HF_TOKEN`, the 4 Phase-1 model datasets.

In [ ]:
RUN_SET = "B"
SETS = {
    "A": [
        dict(ARCH="hybrid_prefixlm",  METHOD="full",     LR=5e-5),
        dict(ARCH="efficient_retnet", METHOD="kd", KD_TEACHER_RUN="hybrid_prefixlm_full"),
        dict(ARCH="hybrid_prefixlm",  METHOD="lorma",    LR=3e-4),
        dict(ARCH="encdec_t5",        METHOD="houlsby",  LR=5e-4),
    ],
    "B": [
        dict(ARCH="hybrid_prefixlm",  METHOD="lora",     LR=5e-4),
        dict(ARCH="hybrid_prefixlm",  METHOD="lorma",    LR=3e-4, RUN_NAME="hybrid_prefixlm_lorma_v2"),
        dict(ARCH="decoder_llama",    METHOD="pfeiffer", LR=5e-4),
        dict(ARCH="hybrid_prefixlm",  METHOD="prefix",   LR=1e-3, N_PREFIX=32),
        dict(ARCH="decoder_llama",    METHOD="prompt",   LR=1e-2, N_PROMPT=32),
        dict(ARCH="hybrid_prefixlm",  METHOD="lora",     LR=5e-4, LONG_DOC="flip",
             RUN_NAME="hybrid_prefixlm_lora_longdoc_ablation"),
    ],
}
EXPERIMENTS = SETS[RUN_SET]
TIME_BUDGET_HOURS = 11.0
MAX_TRAIN_MINUTES = 110

LONG_DOC = "auto"
CONTEXT_SELECT = "select"
MAX_SEGMENTS = 6
SRC_LEN = 704
TGT_LEN = 288
CHARS_PER_TOKEN = 4.6

GEN_BATCH_SIZE = 8
NO_REPEAT_NGRAM = 3
REP_PENALTY = 1.0
MIN_NEW = None
LENGTH_CONTROL = True
LENGTH_SCALE = 1.0
CLEAN_SEGMENTS = True
SEG_MAX_NEW = None
VAL_DOCS = 120
VAL_GEN_DOCS = 60
COMPUTE_BERTSCORE = True
BERTSCORE_MODEL = "roberta-large"

EPOCHS = 4
TRAIN_PAIRS_PER_EPOCH = 12000
BATCH_SIZE = 2
GRAD_ACCUM = 8
EVAL_BATCH_SIZE = 4
LR = None
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.05
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.0
AMP = True
SEED = 42
LOG_EVERY = 50

RANK, ALPHA, PEFT_DROPOUT = 16, 32, 0.05
ADAPTER_DIM = 64
N_PROMPT = 0
N_PREFIX = 0
KD_TEACHER_RUN = None
KD_TEACHER_DIR = None
KD_KEEP_LAYERS = [0, 2, 4, 6, 8, 10]
KD_ALPHA, KD_TEMPERATURE = 0.5, 2.0
SAVE_WEIGHTS = False

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DATA_PREFIX = "Phase_6_Summ"
LOCAL_DATA_DIR = None
SMOKE_TEST = False

DEFAULT_LR = {"full": 5e-5, "kd": 1e-4, "lora": 5e-4, "lorma": 3e-4, "houlsby": 5e-4, "pfeiffer": 5e-4,
              "prompt": 1e-2, "prefix": 1e-3}
BASE_C = {k: v for k, v in dict(globals()).items() if k.isupper() and not k.startswith("_")}

def experiment_config(overrides):
    c = dict(BASE_C, **overrides)
    c["LR"] = overrides.get("LR") or DEFAULT_LR.get(c["METHOD"], 0.0)
    c["RUN_NAME"] = overrides.get("RUN_NAME") or f"{c['ARCH']}_{c['METHOD']}"
    return c

C = experiment_config(EXPERIMENTS[0])
print(f"RUN SET {RUN_SET}")
for _i, _e in enumerate(EXPERIMENTS, 1):
    _c = experiment_config(_e)
    print(f"{_i}. {_c['RUN_NAME']:40s} arch={_c['ARCH']:16s} method={_c['METHOD']:8s} long_doc={_c['LONG_DOC']:6s} lr={_c['LR']}")

In [ ]:
import os, sys, json, glob, time, random, shutil, zipfile, hashlib, math, re, subprocess
from collections import Counter
if not SMOKE_TEST and COMPUTE_BERTSCORE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bert-score'], check=False)
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
if device.type != "cuda" and not SMOKE_TEST:
    raise RuntimeError("NO GPU: this session runs on CPU (training would take > 12 h). In Kaggle: Settings -> Accelerator -> "
                       "GPU T4 x2 (check your weekly GPU quota), then run all cells again.")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

def hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    for name in ("HF_TOKEN", "HF"):
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret(name)
            if tok: return tok.strip()
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN_VALUE = hf_token()
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
    print("Hugging Face token found (length", len(HF_TOKEN_VALUE), ")")
else:
    print("WARNING: no Hugging Face token found. On Kaggle: Add-ons -> Secrets -> add 'HF_TOKEN' (or 'HF') "
          "and switch its toggle ON for this notebook; the course dataset is gated.")

def data_file(rel):
    roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []
    for root in roots:
        p = os.path.join(root, rel)
        if os.path.exists(p): return p
    hits = glob.glob(f"/kaggle/input/**/{DATA_PREFIX}/{rel}", recursive=True)
    if hits: return hits[0]
    from huggingface_hub import hf_hub_download
    assert HF_TOKEN_VALUE, ("No Hugging Face token: add the Kaggle secret HF_TOKEN (or HF), switch it ON for this notebook "
                           "(Add-ons -> Secrets), then run again.")
    return hf_hub_download(HF_DATASET_REPO, f"{DATA_PREFIX}/{rel}", repo_type="dataset", token=HF_TOKEN_VALUE)

def find_model_dir(arch):
    for cfgp in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            if json.load(open(cfgp)).get("arch") == arch and os.path.exists(os.path.join(os.path.dirname(cfgp), "pytorch_model.bin")):
                return os.path.dirname(cfgp)
        except Exception:
            pass
    raise FileNotFoundError(f"no Phase-1 model folder for arch={arch} under /kaggle/input "
                            "(attach the dataset that contains config.json + pytorch_model.bin + legal_bpe_tokenizer.json)")

### Phase-1 code (unchanged)

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_positions, theta=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", torch.cat([freqs.cos(), freqs.cos()], dim=-1), persistent=False)
        self.register_buffer("sin_cached", torch.cat([freqs.sin(), freqs.sin()], dim=-1), persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos_cached[offset:offset + T].to(x.dtype)
        sin = self.sin_cached[offset:offset + T].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return x * cos + torch.cat((-x2, x1), dim=-1) * sin

class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = rope(q, offset), rope(k, offset)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal and attn_mask is None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_gate_up = nn.Linear(n_embd, 2 * hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        gate, up = self.w_gate_up(x).chunk(2, dim=-1)
        return self.w_down(F.silu(gate) * up)

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        x = x + self.drop(self.attn(self.norm1(x), rope, attn_mask, is_causal, cache, offset))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

def init_weights(model, n_layers, std=0.02):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight") or name.endswith("w_down.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layers))

def lm_loss(lm_head, hidden, input_ids, loss_start=0):
    start = max(int(loss_start), 1)
    logits = lm_head(hidden[:, start - 1:-1])
    targets = input_ids[:, start:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

class DecoderOnlyLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        h = self._hidden(ids)
        return self.lm_head(h[:, prompt_ids.size(1) - 1:-1])

def prefix_lm_mask(T, prefix_len, device):
    i = torch.arange(T, device=device)
    mask = (i[None, :] <= i[:, None]) | (i[None, :] < prefix_len)
    return mask[None, None]

class PrefixLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.segment = nn.Embedding(2, d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, prefix_len, caches=None, offset=0, decoding=False):
        B, T = input_ids.shape
        positions = torch.arange(offset, offset + T, device=input_ids.device)
        segments = (positions >= prefix_len).long()
        x = self.drop(self.embed(input_ids) + self.segment(segments)[None])

        if decoding:
            attn_mask, is_causal = None, False
        elif prefix_len >= T:
            attn_mask, is_causal = None, False
        elif prefix_len <= 0:
            attn_mask, is_causal = None, True
        else:
            attn_mask, is_causal = prefix_lm_mask(T, prefix_len, input_ids.device), False

        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, attn_mask=attn_mask, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        return lm_loss(self.lm_head, self._hidden(input_ids, P), input_ids, P)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        P = prompt_ids.size(1)
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, prefix_len=P, caches=caches, offset=0)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": P, "prefix_len": P}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["prefix_len"], state["caches"],
                         offset=state["pos"], decoding=True)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        P = prompt_ids.size(1)
        h = self._hidden(torch.cat([prompt_ids, cont_ids], dim=1), prefix_len=P)
        return self.lm_head(h[:, P - 1:-1])

class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.kv = nn.Linear(n_embd, 2 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, memory, cache=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if cache is not None and "k" in cache:
            k, v = cache["k"], cache["v"]
        else:
            S = memory.size(1)
            k, v = self.kv(memory).view(B, S, 2, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
            if cache is not None:
                cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope):
        x = x + self.drop(self.attn(self.norm1(x), rope, is_causal=False))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

class CrossDecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.self_attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head, dropout)
        self.norm3 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, rope, cache=None, offset=0, is_causal=True):
        self_cache = None if cache is None else cache["self"]
        cross_cache = None if cache is None else cache["cross"]
        x = x + self.drop(self.self_attn(self.norm1(x), rope, is_causal=is_causal, cache=self_cache, offset=offset))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, cross_cache))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x

class EncoderDecoderLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, H = cfg["n_embd"], cfg["n_head"]
        self.bos_id = cfg["bos_token_id"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.enc_blocks = nn.ModuleList(
            [EncoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_enc_layer"])]
        )
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList(
            [CrossDecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_dec_layer"])]
        )
        self.dec_norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, cfg["n_enc_layer"] + cfg["n_dec_layer"])

    def encode(self, src_ids):
        x = self.drop(self.embed(src_ids))
        for block in self.enc_blocks:
            x = block(x, self.rope)
        return self.enc_norm(x)

    def decode(self, tgt_in, memory, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(tgt_in))
        for i, block in enumerate(self.dec_blocks):
            x = block(x, memory, self.rope, None if caches is None else caches[i], offset, is_causal)
        return self.dec_norm(x)

    def _shift_right(self, tgt):
        bos = torch.full_like(tgt[:, :1], self.bos_id)
        return torch.cat([bos, tgt[:, :-1]], dim=1)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        assert 0 < P < input_ids.size(1), "encoder-decoder needs 0 < prefix_len < block size"
        src, tgt = input_ids[:, :P], input_ids[:, P:]
        h = self.decode(self._shift_right(tgt), self.encode(src))
        logits = self.lm_head(h)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

    @torch.no_grad()
    def prefill(self, prompt_ids):
        memory = self.encode(prompt_ids)
        caches = [{"self": {}, "cross": {}} for _ in self.dec_blocks]
        bos = torch.full((prompt_ids.size(0), 1), self.bos_id, dtype=torch.long, device=prompt_ids.device)
        h = self.decode(bos, memory, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"memory": memory, "caches": caches, "pos": 1}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self.decode(token_ids[:, None], state["memory"], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        return self.lm_head(self.decode(self._shift_right(cont_ids), self.encode(prompt_ids)))

class MultiScaleRetention(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkvg = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        gammas = 1.0 - 2.0 ** (-5.0 - torch.arange(n_head, dtype=torch.float32))
        self.register_buffer("log_gamma", torch.log(gammas), persistent=False)

    def _decay_matrix(self, T, device):
        idx = torch.arange(T, device=device, dtype=torch.float32)
        diff = idx[:, None] - idx[None, :]
        decay = torch.exp(diff.clamp(min=0)[None] * self.log_gamma[:, None, None])
        decay = decay.masked_fill(diff[None] < 0, 0.0)
        return decay / decay.sum(dim=-1, keepdim=True).sqrt()

    def forward(self, x, rope, state=None, offset=0):
        B, T, C = x.shape
        q, k, v, g = self.qkvg(x).split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = rope(q, offset), rope(k, offset)
        k = k * self.head_dim ** -0.5

        if state is not None and "S" in state:
            with torch.autocast(device_type=x.device.type, enabled=False):
                gamma = self.log_gamma.exp().view(1, -1, 1, 1)
                state["S"] = gamma * state["S"] + k.float().transpose(-1, -2) @ v.float()
                state["scale"] = gamma * state["scale"] + 1.0
                y = ((q.float() @ state["S"]) / state["scale"].sqrt()).to(v.dtype)
        else:
            decay = self._decay_matrix(T, x.device).to(q.dtype)
            y = ((q @ k.transpose(-1, -2)) * decay) @ v
            if state is not None:
                with torch.autocast(device_type=x.device.type, enabled=False):
                    pos = torch.arange(T, device=x.device, dtype=torch.float32)
                    w = torch.exp((T - 1 - pos)[None, :] * self.log_gamma[:, None])
                    state["S"] = torch.einsum("bhtd,ht,bhte->bhde", k.float(), w, v.float())
                    state["scale"] = w.sum(dim=-1).view(1, -1, 1, 1)

        y = y.transpose(1, 2).float()
        y = y * torch.rsqrt(y.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
        y = y.reshape(B, T, C).to(g.dtype)
        return self.proj(F.silu(g) * y)

class FeedForward(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_in = nn.Linear(n_embd, hidden)
        self.w_down = nn.Linear(hidden, n_embd)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_in(x)))

class RetNetBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.retention = MultiScaleRetention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, state=None, offset=0):
        x = x + self.drop(self.retention(self.norm1(x), rope, state, offset))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class RetNetLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList([RetNetBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, states=None, offset=0):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, None if states is None else states[i], offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        states = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, states, offset=0)
        return self.lm_head(h[:, -1]), {"states": states, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["states"], offset=state["pos"])
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        return self.lm_head(self._hidden(ids)[:, prompt_ids.size(1) - 1:-1])

ARCH_CLASSES = {'decoder_llama': DecoderOnlyLM, 'hybrid_prefixlm': PrefixLM, 'encdec_t5': EncoderDecoderLM, 'efficient_retnet': RetNetLM}

### Fine-tuning modules (LoRA, LoRMA, adapters; from Phases 2-5)

In [ ]:
import math, time, json, os
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

class RSLoRALinear(LoRALinear):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__(base, r, alpha, dropout)
        self.scale = alpha / math.sqrt(r)

class IA3Linear(nn.Module):
    def __init__(self, base, mode, n_split=3):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "out_kv":
            d = base.out_features // n_split
            self.d = d
            self.l_kv = nn.Parameter(torch.ones(2 * d))
        else:
            self.l_in = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.l_in)
        y = self.base(x)
        d = self.d
        scale = torch.cat([torch.ones(d, device=y.device, dtype=y.dtype), self.l_kv.to(y.dtype),
                           torch.ones(y.size(-1) - 3 * d, device=y.device, dtype=y.dtype)])
        return y * scale

class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.base(x)
        return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale

class AdapterLinear(nn.Module):
    def __init__(self, base, bottleneck=64):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, bottleneck)
        self.up = nn.Linear(bottleneck, base.out_features)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x):
        return self.base(x) + self.bias

ATTN_LINEARS = ("qkv", "proj", "qkvg", "q", "kv")
FFN_LINEARS = ("w_gate_up", "w_down", "w_in")
OUT_PROJ = ("proj", "w_down")

def _replace_linears(module, names, make):
    n = 0
    for parent in list(module.modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names:
                setattr(parent, cname, make(child)); n += 1
    return n

def backbone_blocks(arch, bb):
    return list(bb.enc_blocks) if arch == "encdec_t5" else list(bb.blocks)

class NERModel(nn.Module):
    def __init__(self, arch, backbone, n_tags, context="bidir", head="linear", n_prompt=0, dropout=0.1):
        super().__init__()
        self.arch, self.bb, self.context = arch, backbone, context
        self.head_out = n_tags
        d = backbone.cfg["n_embd"]
        self.drop = nn.Dropout(dropout)
        if head == "mlp":
            self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, n_tags))
        else:
            self.head = nn.Linear(d, n_tags)
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt > 0 else None
        if hasattr(backbone, "lm_head"):
            backbone.lm_head = nn.Identity()
        if arch == "encdec_t5":
            backbone.dec_blocks = nn.ModuleList(); backbone.dec_norm = nn.Identity()

    def _mask(self, key_ok, T, causal):
        m = key_ok[:, None, None, :].expand(-1, 1, T, T)
        if causal:
            m = m & torch.ones(T, T, dtype=torch.bool, device=key_ok.device).tril()[None, None]
        eye = torch.eye(T, dtype=torch.bool, device=key_ok.device)[None, None]
        return m | eye

    def hidden(self, ids, attn):
        bb, arch = self.bb, self.arch
        B, T = ids.shape
        x = bb.embed(ids)
        ok = attn.bool()
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = torch.cat([x, x], dim=1); ok = torch.cat([ok, ok], dim=1)
        if arch == "hybrid_prefixlm":
            x = x + bb.segment(torch.zeros(x.size(1), dtype=torch.long, device=x.device))[None]
        P = 0
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt[None].expand(B, -1, -1).to(x.dtype), x], dim=1)
            ok = torch.cat([torch.ones(B, P, dtype=torch.bool, device=x.device), ok], dim=1)
        x = bb.drop(x)
        L = x.size(1)
        if arch == "efficient_retnet":
            for blk in bb.blocks:
                x = blk(x, bb.rope)
            x = bb.norm(x)
        elif arch == "encdec_t5":
            mask = self._mask(ok, L, causal=False)
            for blk in bb.enc_blocks:
                x = x + blk.drop(blk.attn(blk.norm1(x), bb.rope, attn_mask=mask, is_causal=False))
                x = x + blk.drop(blk.mlp(blk.norm2(x)))
            x = bb.enc_norm(x)
        else:
            causal = self.context in ("causal", "echo") and arch == "decoder_llama"
            mask = self._mask(ok, L, causal=causal)
            for blk in bb.blocks:
                x = blk(x, bb.rope, attn_mask=mask, is_causal=False)
            x = bb.norm(x)
        x = x[:, P:]
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = x[:, T:]
        return x

    def forward(self, ids, attn):
        return self.head(self.drop(self.hidden(ids, attn)))

def load_backbone(model_dir, arch_classes, map_location="cpu"):
    cfg = json.load(open(os.path.join(model_dir, "config.json")))
    bb = arch_classes[cfg["arch"]](cfg)
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location=map_location)
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    missing = [k for k in missing if not k.endswith("lm_head.weight")]
    assert not missing and not unexpected, (missing, unexpected)
    return cfg, bb

def keep_layers(arch, bb, keep):
    if arch == "encdec_t5":
        bb.enc_blocks = nn.ModuleList([bb.enc_blocks[i] for i in keep]); bb.cfg["n_enc_layer"] = len(keep)
    else:
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep]); bb.cfg["n_layer"] = len(keep)
    return bb

def apply_method(model, method, C):
    arch, bb = model.arch, model.bb
    info = {}
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "partial":
            blocks = backbone_blocks(arch, bb)
            for blk in blocks[-C["PARTIAL_TOP_K"]:]:
                for p in blk.parameters(): p.requires_grad = True
            norm = bb.enc_norm if arch == "encdec_t5" else bb.norm
            for p in norm.parameters(): p.requires_grad = True
        elif method in ("lora", "lorma", "rslora"):
            cls = {"lora": LoRALinear, "lorma": LoRMALinear, "rslora": RSLoRALinear}[method]
            targets = ATTN_LINEARS + (FFN_LINEARS if C["PEFT_TARGET_FFN"] else ())
            info["wrapped"] = _replace_linears(bb, targets, lambda l: cls(l, C["RANK"], C["ALPHA"], C["PEFT_DROPOUT"]))
        elif method == "adapter":
            info["wrapped"] = _replace_linears(bb, OUT_PROJ, lambda l: AdapterLinear(l, C["ADAPTER_DIM"]))
        elif method == "bitfit":
            info["wrapped"] = _replace_linears(bb, ATTN_LINEARS + FFN_LINEARS, BiasLinear)
            for n, p in bb.named_parameters():
                if n.endswith("bias") or "norm" in n: p.requires_grad = True
        elif method == "gradual":
            pass
        elif method == "ia3":
            n = _replace_linears(bb, ("qkv",), lambda l: IA3Linear(l, "out_kv", 3))
            n += _replace_linears(bb, ("qkvg",), lambda l: IA3Linear(l, "out_kv", 4))
            n += _replace_linears(bb, ("w_down",), lambda l: IA3Linear(l, "in"))
            info["wrapped"] = n
        elif method == "prompt":
            pass
        else:
            raise ValueError(method)
        for p in model.head.parameters(): p.requires_grad = True
        if model.prompt is not None: model.prompt.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    info.update(trainable_params=n_train, total_params=n_total, trainable_pct=100.0 * n_train / n_total)
    return info

### Summarisation: long documents, model, PEFT, KD, training, generation, metrics

In [ ]:
import json, os, re, math, time, random, hashlib
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def parse_docs(obj):
    items = obj.items() if isinstance(obj, dict) and "document" not in obj and "text" not in obj else \
        [(None, x) for x in (obj if isinstance(obj, list) else [obj])]
    out = []
    for k, d in items:
        if isinstance(d, str): d = {"document": d}
        doc = d.get("document", d.get("text", d.get("judgment", "")))
        if isinstance(doc, list): doc = "\n".join(map(str, doc))
        summ = d.get("summary", d.get("abstract"))
        if isinstance(summ, list): summ = "\n".join(map(str, summ))
        out.append({"id": d.get("id", d.get("doc_id", k)), "document": str(doc),
                    "summary": None if summ is None else str(summ)})
    return out

_SENT_RE = re.compile(r"(?<=[.?!;])\s+(?=[A-Z0-9(\"'])|\n{2,}")
def split_sentences(text):
    sents = [s.strip() for s in _SENT_RE.split(re.sub(r"[ \t]+", " ", text)) if s and s.strip()]
    out = []
    for s in sents:
        while len(s) > 600:
            cut = s.rfind(" ", 0, 400); cut = cut if cut > 100 else 400
            out.append(s[:cut]); s = s[cut:].strip()
        if s: out.append(s)
    return out

_WORD_RE = re.compile(r"[a-z0-9]+")
STOP = set("the of and to in a is that for be by on as was with it this which are or at from an not has have had been "
           "were his her he she they their its any shall may such also there said would no into these those under upon "
           "being who whom than other so if all them can will only one two".split())
def words(s):
    return [w for w in _WORD_RE.findall(s.lower()) if w not in STOP and len(w) > 1]

def select_context(sents, budget_chars, mode):
    total = sum(len(s) + 1 for s in sents)
    if total <= budget_chars or mode == "lead":
        keep, n = [], 0
        for s in sents:
            if n + len(s) > budget_chars and keep: break
            keep.append(s); n += len(s) + 1
        return keep
    chosen, used = set(), 0
    def take(i):
        nonlocal used
        if i in chosen or used + len(sents[i]) > budget_chars: return False
        chosen.add(i); used += len(sents[i]) + 1; return True
    i = 0
    while i < len(sents) and used < 0.20 * budget_chars: take(i); i += 1
    j, tail = len(sents) - 1, 0
    while j > i and tail < 0.15 * budget_chars:
        if take(j): tail += len(sents[j]) + 1
        j -= 1
    tf = Counter(w for s in sents for w in set(words(s)))
    score = {k: sum(tf[w] for w in set(words(s))) / math.sqrt(len(words(s)) + 1) for k, s in enumerate(sents) if k not in chosen}
    for k in sorted(score, key=lambda k: -score[k]):
        if used >= budget_chars: break
        take(k)
    return [sents[k] for k in sorted(chosen)]

def segment_document(sents, n_seg):
    if n_seg <= 1: return [sents]
    total = sum(len(s) for s in sents); bounds, acc, segs, cur = [total * (k + 1) / n_seg for k in range(n_seg)], 0, [], []
    for s in sents:
        cur.append(s); acc += len(s)
        if len(segs) < n_seg - 1 and acc >= bounds[len(segs)]: segs.append(cur); cur = []
    segs.append(cur)
    return [s for s in segs if s]

def align_summary(summary_sents, segments):
    seg_words = [Counter(w for s in seg for w in words(s)) for seg in segments]
    out = [[] for _ in segments]
    for s in summary_sents:
        ws = set(words(s))
        if not ws: out[0].append(s); continue
        best = max(range(len(segments)), key=lambda k: (sum(1 for w in ws if seg_words[k][w]) / len(ws), -k))
        out[best].append(s)
    return [" ".join(x) for x in out]

def n_segments(doc_chars, C):
    return int(min(C["MAX_SEGMENTS"], max(1, round(doc_chars / (C["SRC_LEN"] * C["CHARS_PER_TOKEN"])))))

def build_pairs(docs, tok, C, with_targets=True):
    budget = C["SRC_LEN"] * C["CHARS_PER_TOKEN"]
    pairs = []
    for d in docs:
        sents = split_sentences(d["document"]) or [d["document"][:budget] or "."]
        segs = segment_document(sents, n_segments(len(d["document"]), C)) if C["LONG_DOC"] == "chunked" else [sents]
        tgts = [None] * len(segs)
        if with_targets and d["summary"] is not None:
            ss = split_sentences(d["summary"]) or [d["summary"]]
            tgts = align_summary(ss, segs) if len(segs) > 1 else [" ".join(ss)]
        for k, (seg, tgt) in enumerate(zip(segs, tgts)):
            if with_targets and not (tgt or "").strip(): continue
            ctx = select_context(seg, budget, C["CONTEXT_SELECT"])
            src = tok.encode(" ".join(ctx))[: C["SRC_LEN"]] or [tok.eos_token_id]
            p = {"doc_id": d["id"], "seg": k, "n_seg": len(segs), "src": src}
            if tgt is not None:
                p["tgt"] = tok.encode(tgt)[: C["TGT_LEN"] - 1] + [tok.eos_token_id]
            pairs.append(p)
    return pairs

class PrefixEncoder(nn.Module):
    def __init__(self, n_layers, n_head, head_dim, n_prefix, hidden=512):
        super().__init__()
        d = n_head * head_dim
        self.n_layers, self.n_head, self.head_dim, self.n_prefix = n_layers, n_head, head_dim, n_prefix
        self.emb = nn.Parameter(torch.randn(n_prefix, d) * 0.02)
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.Tanh(), nn.Linear(hidden, n_layers * 2 * d))
    def forward(self, B, dtype):
        kv = self.mlp(self.emb).view(self.n_prefix, self.n_layers, 2, self.n_head, self.head_dim)
        kv = kv.permute(1, 2, 3, 0, 4).to(dtype)
        return [(kv[l, 0][None].expand(B, -1, -1, -1), kv[l, 1][None].expand(B, -1, -1, -1)) for l in range(self.n_layers)]

class SummModel(nn.Module):
    def __init__(self, arch, bb, n_prompt=0, n_prefix=0):
        super().__init__()
        self.arch, self.bb = arch, bb
        d = bb.cfg["n_embd"]
        self.prompt = nn.Parameter(bb.embed.weight[torch.randint(256, min(5000, bb.embed.weight.size(0)), (n_prompt,))].detach().clone()) if n_prompt else None
        self.prefix = None
        if n_prefix:
            assert arch in ("decoder_llama", "hybrid_prefixlm"), "prefix tuning is implemented for the DecoderBlock models"
            att = bb.blocks[0].attn
            self.prefix = PrefixEncoder(len(bb.blocks), att.n_head, att.head_dim, n_prefix)
        self.head = nn.Identity()
        if arch == "encdec_t5" and self.prompt is not None:
            raise ValueError("prompt tuning is used with the decoder-only / prefix models here")

    @property
    def n_virtual(self):
        return 0 if self.prompt is None else self.prompt.size(0)

    def max_len(self):
        return int(self.bb.cfg["max_positions"])

    def _stack(self, ids, prefix_len, caches=None, offset=0, decoding=False):
        bb = self.bb; B, T = ids.shape
        x = bb.embed(ids)
        if self.prompt is not None and not decoding:
            x = torch.cat([self.prompt[None].expand(B, -1, -1).to(x.dtype), x], 1); T = x.size(1)
        if self.arch == "hybrid_prefixlm":
            pos = torch.arange(offset, offset + T, device=ids.device)
            x = x + bb.segment((pos >= prefix_len).long())[None]
        x = bb.drop(x)
        npk = 0 if self.prefix is None else self.prefix.n_prefix
        if decoding:
            mask = None
        else:
            i = torch.arange(T, device=ids.device)
            m = i[None, :] <= i[:, None]
            if self.arch == "hybrid_prefixlm": m = m | (i[None, :] < prefix_len)
            if npk: m = torch.cat([torch.ones(T, npk, dtype=torch.bool, device=ids.device), m], 1)
            mask = m[None, None]
        pkv = self.prefix(B, x.dtype) if (self.prefix is not None and not decoding) else None
        for l, blk in enumerate(bb.blocks):
            cache = caches[l] if caches is not None else ({} if pkv is not None else None)
            if pkv is not None: cache["k"], cache["v"] = pkv[l]
            x = blk(x, bb.rope, attn_mask=mask, is_causal=False, cache=cache, offset=offset)
        return bb.norm(x)

    def cont_logits(self, src, cont):
        bb, P = self.bb, src.size(1)
        if self.arch == "encdec_t5":
            mem = bb.encode(src)
            return bb.lm_head(bb.decode(bb._shift_right(cont), mem))
        ids = torch.cat([src, cont[:, :-1]], 1)
        if self.arch == "efficient_retnet":
            h = bb._hidden(ids)
            return bb.lm_head(h[:, P - 1:])
        h = self._stack(ids, prefix_len=P + self.n_virtual)
        return bb.lm_head(h[:, self.n_virtual + P - 1:])

    @torch.no_grad()
    def prefill(self, src):
        if self.arch in ("encdec_t5", "efficient_retnet"):
            return self.bb.prefill(src)
        caches = [dict() for _ in self.bb.blocks]
        P = src.size(1) + self.n_virtual
        h = self._stack(src, prefix_len=P, caches=caches)
        return self.bb.lm_head(h[:, -1]), {"caches": caches, "pos": P, "prefix_len": P}

    @torch.no_grad()
    def decode_step(self, tok_ids, state):
        if self.arch in ("encdec_t5", "efficient_retnet"):
            return self.bb.decode_step(tok_ids, state)
        h = self._stack(tok_ids[:, None], state["prefix_len"], state["caches"], offset=state["pos"], decoding=True)
        state["pos"] += 1
        return self.bb.lm_head(h[:, -1]), state

def apply_peft(model, method, C):
    bb, info = model.bb, {}
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method in ("lora", "lorma"):
            cls = LoRALinear if method == "lora" else LoRMALinear
            info["wrapped"] = _replace_linears(bb, ATTN_LINEARS + FFN_LINEARS, lambda l: cls(l, C["RANK"], C["ALPHA"], C["PEFT_DROPOUT"]))
        elif method == "houlsby":
            info["wrapped"] = _replace_linears(bb, ("proj", "w_down"), lambda l: AdapterLinear(l, C["ADAPTER_DIM"]))
        elif method == "pfeiffer":
            info["wrapped"] = _replace_linears(bb, ("w_down",), lambda l: AdapterLinear(l, C["ADAPTER_DIM"] * 2))
        elif method in ("prompt", "prefix"):
            pass
        else:
            raise ValueError(method)
        for m in (model.prompt, model.prefix):
            if m is None: continue
            for p in ([m] if isinstance(m, nn.Parameter) else m.parameters()): p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    info.update(trainable_params=n_train, total_params=n_total, trainable_pct=100.0 * n_train / n_total)
    return info

def make_batches(pairs, batch_size, rng=None):
    by_len = {}
    for i, p in enumerate(pairs): by_len.setdefault(len(p["src"]), []).append(i)
    batches = []
    for L, idx in by_len.items():
        if rng: rng.shuffle(idx)
        batches += [idx[i:i + batch_size] for i in range(0, len(idx), batch_size)]
    if rng: rng.shuffle(batches)
    return batches

def collate(pairs, idx, pad_id, device):
    src = torch.tensor([pairs[i]["src"] for i in idx], dtype=torch.long)
    Tc = max(len(pairs[i]["tgt"]) for i in idx)
    cont = torch.full((len(idx), Tc), pad_id, dtype=torch.long); y = torch.full((len(idx), Tc), -100, dtype=torch.long)
    for r, i in enumerate(idx):
        t = pairs[i]["tgt"]; cont[r, :len(t)] = torch.tensor(t); y[r, :len(t)] = torch.tensor(t)
    return src.to(device), cont.to(device), y.to(device)

def seq_kd_loss(s_logits, t_logits, y, T):
    m = y != -100
    s = F.log_softmax(s_logits[m].float() / T, -1); t = F.softmax(t_logits[m].float() / T, -1)
    return F.kl_div(s, t, reduction="batchmean") * T * T

def train(model, train_pairs, val_pairs, C, pad_id, device, teacher=None, log=print):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=C["LR"], weight_decay=C["WEIGHT_DECAY"])
    rng = random.Random(C["SEED"])
    n_ep = min(len(train_pairs), C["TRAIN_PAIRS_PER_EPOCH"])
    n_batches = len(make_batches(train_pairs[:n_ep], C["BATCH_SIZE"]))
    steps_per_epoch = math.ceil(n_batches / C["GRAD_ACCUM"])
    total = steps_per_epoch * C["EPOCHS"]; warm = int(C["WARMUP_FRAC"] * total)
    amp = C["AMP"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best, best_state, hist, step, t0 = float("inf"), None, [], 0, time.time()
    time_up = False
    for ep in range(1, C["EPOCHS"] + 1):
        if time_up: break
        model.train()
        sub_idx = list(range(len(train_pairs))); rng.shuffle(sub_idx); sub = [train_pairs[i] for i in sub_idx[:n_ep]]
        run, micro, ntok = 0.0, 0, 0
        for b in make_batches(sub, C["BATCH_SIZE"], rng):
            src, cont, y = collate(sub, b, pad_id, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                lg = model.cont_logits(src, cont)
                loss = F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), y.reshape(-1), ignore_index=-100,
                                       label_smoothing=C["LABEL_SMOOTHING"])
                if teacher is not None:
                    with torch.no_grad(): tl = teacher.cont_logits(src, cont)
                    loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * seq_kd_loss(lg, tl, y, C["KD_TEMPERATURE"])
            scaler.scale(loss / C["GRAD_ACCUM"]).backward(); micro += 1; run += loss.item(); ntok += int((y != -100).sum())
            if micro % C["GRAD_ACCUM"] == 0:
                lr = C["LR"] * (step / max(1, warm) if step < warm else max(0.0, (total - step) / max(1, total - warm)))
                for g in opt.param_groups: g["lr"] = lr
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % C["LOG_EVERY"] == 0:
                    log(f"  epoch {ep} step {step}/{total} loss {run / micro:.4f} ({time.time() - t0:.0f}s)")
                if (time.time() - t0) / 60 > C["MAX_TRAIN_MINUTES"]:
                    log(f"  time cap of {C['MAX_TRAIN_MINUTES']} min reached at step {step}/{total}: stopping this run")
                    time_up = True; break
        vl = eval_loss(model, val_pairs, C, pad_id, device)
        hist.append({"epoch": ep, "train_loss": run / max(1, micro), "val_loss": vl, "val_ppl": math.exp(min(vl, 20)),
                     "time_s": time.time() - t0})
        log(f"epoch {ep}: train loss {run / max(1, micro):.4f} | val loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.1f}) ({time.time() - t0:.0f}s)")
        if vl < best:
            best = vl; trainable = {n for n, p in model.named_parameters() if p.requires_grad}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in trainable}
    if best_state is not None: model.load_state_dict(best_state, strict=False)
    return {"best_val_loss": best, "history": hist, "train_time_s": time.time() - t0, "steps": step}

@torch.no_grad()
def eval_loss(model, pairs, C, pad_id, device):
    model.eval(); tot, n = 0.0, 0
    for b in make_batches(pairs, C["EVAL_BATCH_SIZE"]):
        src, cont, y = collate(pairs, b, pad_id, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=C["AMP"] and device.type == "cuda"):
            lg = model.cont_logits(src, cont)
        tot += F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), y.reshape(-1), ignore_index=-100, reduction="sum").item()
        n += int((y != -100).sum())
    return tot / max(1, n)

@torch.no_grad()
def generate_batch(model, src, max_new, min_new, eos_id, no_repeat=3, rep_penalty=1.0):
    B = src.size(0)
    max_r = list(max_new) if isinstance(max_new, (list, tuple)) else [max_new] * B
    min_r = list(min_new) if isinstance(min_new, (list, tuple)) else [min_new] * B
    max_new = max(max_r)
    with torch.autocast(device_type=src.device.type, dtype=torch.float16, enabled=src.device.type == "cuda"):
        logits, state = model.prefill(src)
    out = [[] for _ in range(B)]; done = [False] * B
    for t in range(max_new):
        lg = logits.float()
        for r in range(B):
            if t < min_r[r]: lg[r, eos_id] = -float("inf")
            if t >= max_r[r]: lg[r, :] = -float("inf"); lg[r, eos_id] = 0.0
        for r in range(B):
            if done[r]: continue
            seq = out[r]
            if rep_penalty != 1.0 and seq:
                ix = torch.tensor(sorted(set(seq[-64:])), device=lg.device)
                v = lg[r, ix]; lg[r, ix] = torch.where(v > 0, v / rep_penalty, v * rep_penalty)
            if no_repeat and len(seq) >= no_repeat - 1:
                key = tuple(seq[-(no_repeat - 1):]) if no_repeat > 1 else ()
                banned = [seq[i + no_repeat - 1] for i in range(len(seq) - no_repeat + 1) if tuple(seq[i:i + no_repeat - 1]) == key]
                if banned: lg[r, banned] = -float("inf")
        nxt = lg.argmax(-1)
        for r in range(B):
            if done[r]: continue
            tk = int(nxt[r])
            if tk == eos_id: done[r] = True
            else: out[r].append(tk)
        if all(done): break
        with torch.autocast(device_type=src.device.type, dtype=torch.float16, enabled=src.device.type == "cuda"):
            logits, state = model.decode_step(nxt, state)
    return out

def dedupe_sentences(text, thr=0.8):
    keep, seen = [], []
    for s in split_sentences(text):
        w = set(words(s))
        if w and any(len(w & v) / len(w | v) >= thr for v in seen): continue
        keep.append(s); seen.append(w)
    return " ".join(keep)

_END_RE = re.compile(r"[.!?\]\)]\s*$")
def clean_segment(text, max_frag=0.4):
    t = text.strip().lstrip(",.;: ").strip()
    if t and not _END_RE.search(t):
        cut = max(t.rfind(". "), t.rfind("? "), t.rfind("! "))
        if cut > 0 and (len(t) - cut) / len(t) <= max_frag: t = t[:cut + 1]
    return t

def fit_length_model(docs, tok_per_word):
    x = np.log([max(1, len(d["document"].split())) for d in docs]); y = np.log([max(1, len(d["summary"].split())) for d in docs])
    b, a = np.polyfit(x, y, 1)
    resid = y - (a + b * x)
    return {"a": float(a), "b": float(b), "resid_sd": float(resid.std()), "tok_per_word": float(tok_per_word)}

def length_budget(doc, n_seg, C):
    L = C.get("LENGTH_MODEL")
    if not C.get("LENGTH_CONTROL") or not L:
        return C["MIN_NEW"], (C["TGT_LEN"] if C["LONG_DOC"] == "single" else C["SEG_MAX_NEW"])
    words = math.exp(L["a"] + L["b"] * math.log(max(1, len(doc["document"].split())))) * C["LENGTH_SCALE"]
    per_seg = words * L["tok_per_word"] / n_seg
    hi = int(min(C["TGT_LEN"], max(48, 1.15 * per_seg)))
    lo = int(min(hi - 8, max(16, 0.85 * per_seg)))
    return lo, hi

def summarise(model, docs, tok, C, device, log=print):
    model.eval()
    pairs = build_pairs(docs, tok, C, with_targets=False)
    texts = {}
    t0 = time.time()
    by_id = {d["id"]: d for d in docs}
    for bi, b in enumerate(make_batches(pairs, C["GEN_BATCH_SIZE"])):
        src = torch.tensor([pairs[i]["src"] for i in b], dtype=torch.long, device=device)
        budgets = [length_budget(by_id[pairs[i]["doc_id"]], pairs[i]["n_seg"], C) for i in b]
        outs = generate_batch(model, src, max_new=[h for _, h in budgets], min_new=[l for l, _ in budgets],
                              eos_id=tok.eos_token_id, no_repeat=C["NO_REPEAT_NGRAM"], rep_penalty=C["REP_PENALTY"])
        for i, o in zip(b, outs):
            t = tok.decode(o).strip()
            texts[(pairs[i]["doc_id"], pairs[i]["seg"])] = clean_segment(t) if C.get("CLEAN_SEGMENTS") else t
        if bi % 20 == 0: log(f"  generated {len(texts)}/{len(pairs)} segment summaries ({time.time() - t0:.0f}s)")
    res = {}
    for d in docs:
        parts = [texts[k] for k in sorted(k for k in texts if k[0] == d["id"])]
        s = dedupe_sentences(" ".join(p for p in parts if p))
        if not s.strip():
            s = " ".join(split_sentences(d["document"])[:3])
        res[d["id"]] = s
    return res

def _toks(s): return _WORD_RE.findall(s.lower())
def _ngrams(t, n): return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))
def _f(p, r): return 2 * p * r / (p + r) if p + r else 0.0
def rouge_n(ref, hyp, n):
    R, H = _ngrams(_toks(ref), n), _ngrams(_toks(hyp), n); ov = sum((R & H).values())
    return _f(ov / max(1, sum(H.values())), ov / max(1, sum(R.values())))
def rouge_l(ref, hyp):
    a, b = _toks(ref), _toks(hyp)
    if not a or not b: return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b): cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]; return _f(l / len(b), l / len(a))

def rep_rate(s, n=3):
    g = _ngrams(_toks(s), n); tot = sum(g.values())
    return 0.0 if not tot else 1 - len(g) / tot

def novel_rate(src, s, n=2):
    S = set(_ngrams(_toks(src), n)); H = _ngrams(_toks(s), n); tot = sum(H.values())
    return 0.0 if not tot else sum(c for g, c in H.items() if g not in S) / tot

def bertscore(refs, hyps, C, device):
    if not C["COMPUTE_BERTSCORE"]: return None
    try:
        from bert_score import score
        P, R, Fv = score(hyps, refs, lang="en", model_type=C["BERTSCORE_MODEL"], batch_size=8,
                         device=str(device), verbose=False)
        return float(Fv.mean())
    except Exception as e:
        print("BERTScore not computed:", type(e).__name__, str(e)[:200]); return None

def evaluate_summaries(docs, preds, C, device):
    refs = [d["summary"] for d in docs]; hyps = [preds[d["id"]] for d in docs]
    out = {"rouge1": float(np.mean([rouge_n(r, h, 1) for r, h in zip(refs, hyps)])),
           "rouge2": float(np.mean([rouge_n(r, h, 2) for r, h in zip(refs, hyps)])),
           "rougeL": float(np.mean([rouge_l(r, h) for r, h in zip(refs, hyps)])),
           "pred_words": float(np.mean([len(_toks(h)) for h in hyps])), "ref_words": float(np.mean([len(_toks(r)) for r in refs])),
           "rep3": float(np.mean([rep_rate(h) for h in hyps])),
           "novel2": float(np.mean([novel_rate(d["document"], h) for d, h in zip(docs, hyps)]))}
    out["bertscore_f1"] = bertscore(refs, hyps, C, device)
    out["length_curve"] = {f"{int(f * 100)}%": {"rouge1": float(np.mean([rouge_n(r, " ".join(_toks(h)[: max(1, int(len(_toks(h)) * f))]), 1)
                                                                         for r, h in zip(refs, hyps)])),
                                                "rougeL": float(np.mean([rouge_l(r, " ".join(_toks(h)[: max(1, int(len(_toks(h)) * f))]))
                                                                         for r, h in zip(refs, hyps)]))}
                           for f in (0.25, 0.5, 0.75, 1.0)}
    return out

### 1. Data

In [ ]:
if SMOKE_TEST:
    _rs = random.Random(0)
    _facts = ["The appellant was convicted under Section 302 IPC.", "The High Court confirmed the conviction.",
              "The witnesses turned hostile during the trial.", "The prosecution relied on circumstantial evidence.",
              "The court examined the dying declaration.", "The appeal was filed against the order of the tribunal.",
              "The respondent claimed compensation for the land.", "The State argued that the notification was valid."]
    def _d(i, with_summary=True):
        f = _rs.sample(_facts, 4)
        doc = " ".join(f"{s} Details of paragraph {k} are discussed at length here." for k, s in enumerate(f * 6))
        return {"id": f"doc_{i}", "document": doc, **({"summary": " ".join(f[:3])} if with_summary else {})}
    all_train = parse_docs([_d(i) for i in range(48)]); test_docs = parse_docs([_d(900 + i, False) for i in range(6)])
else:
    all_train = parse_docs(load_json(data_file("Train/train.json")))
    test_docs = parse_docs(load_json(data_file("Test/test.json")))

_h = lambda x: int(hashlib.md5(str(x).encode()).hexdigest(), 16)
_order = sorted(all_train, key=lambda d: _h(d["id"]))
n_val = min(VAL_DOCS, max(2, len(all_train) // 10))
val_docs, train_docs = _order[:n_val], _order[n_val:]
val_gen_docs = val_docs[:min(VAL_GEN_DOCS, len(val_docs))]
assert len({d["id"] for d in test_docs}) == len(test_docs), "duplicate test ids"

def _stats(name, docs, field):
    L = sorted(len(d[field].split()) for d in docs if d[field])
    return f"{name} {field}: {len(L)} | words median {L[len(L)//2]:,}, 10th {L[len(L)//10]:,}, 90th {L[int(len(L)*.9)]:,}, max {L[-1]:,}"
print(f"train {len(train_docs)} | validation {len(val_docs)} (generated: {len(val_gen_docs)}) | test {len(test_docs)}")
for n, ds in (("train", train_docs), ("test", test_docs)): print(_stats(n, ds, "document"))
print(_stats("train", train_docs, "summary"))
_ratio = np.median([len(d["summary"].split()) / max(1, len(d["document"].split())) for d in train_docs])
print(f"median compression (summary words / document words): {_ratio:.3f}")

if SMOKE_TEST:
    tokenizer = LegalBPETokenizer.train([d["document"] + " " + d["summary"] for d in all_train[:10]], vocab_size=400, verbose=False)
else:
    MODEL_PATH = find_model_dir(C["ARCH"]); print("tokenizer from", MODEL_PATH)
    tokenizer = LegalBPETokenizer.from_pretrained(MODEL_PATH)
_s = [d["summary"] for d in train_docs[:200]]
TOK_PER_WORD = sum(len(tokenizer.encode(s)) for s in _s) / max(1, sum(len(s.split()) for s in _s))
med_summ_tok = np.median([len(d["summary"].split()) for d in train_docs]) * TOK_PER_WORD
AUTO_LONG_DOC = "chunked" if med_summ_tok > 1.5 * TGT_LEN else "single"
print(f"tokens per word {TOK_PER_WORD:.2f} | median reference summary ~{med_summ_tok:.0f} tokens vs TGT_LEN {TGT_LEN} "
      f"-> LONG_DOC auto = {AUTO_LONG_DOC}")

LENGTH_MODEL = fit_length_model(train_docs, TOK_PER_WORD)
_pred = [math.exp(LENGTH_MODEL["a"] + LENGTH_MODEL["b"] * math.log(max(1, len(d["document"].split())))) for d in val_docs]
_true = [len(d["summary"].split()) for d in val_docs]
print(f"length model: log(summary words) = {LENGTH_MODEL['a']:.2f} + {LENGTH_MODEL['b']:.2f} log(document words) | "
      f"validation: corr(pred, true) {np.corrcoef(np.log(_pred), np.log(_true))[0, 1]:.2f}, median |error| "
      f"{np.median(np.abs(np.array(_pred) - np.array(_true)) / np.array(_true)):.0%}")

def resolve(c):
    c = dict(c)
    c["LENGTH_MODEL"] = LENGTH_MODEL
    if c["LONG_DOC"] == "auto": c["LONG_DOC"] = AUTO_LONG_DOC
    elif c["LONG_DOC"] == "flip": c["LONG_DOC"] = "single" if AUTO_LONG_DOC == "chunked" else "chunked"
    if SMOKE_TEST: c["SRC_LEN"], c["TGT_LEN"] = 96, 40
    if c["SEG_MAX_NEW"] is None: c["SEG_MAX_NEW"] = c["TGT_LEN"]
    return c

PAIR_CACHE = {}
def get_pairs(c):
    key = (c["LONG_DOC"], c["CONTEXT_SELECT"], c["SRC_LEN"], c["TGT_LEN"], c["MAX_SEGMENTS"])
    if key not in PAIR_CACHE:
        t0 = time.time()
        tr, va = build_pairs(train_docs, tokenizer, c), build_pairs(val_docs, tokenizer, c)
        tl = sorted(len(p["tgt"]) for p in tr)
        PAIR_CACHE[key] = (tr, va, int(np.median(tl)))
        print(f"pairs {key}: train {len(tr)}, val {len(va)} | target tokens median {tl[len(tl)//2]}, "
              f"truncated at TGT_LEN: {np.mean([t >= c['TGT_LEN'] for t in tl]):.0%} | "
              f"segments per doc {len(tr) / max(1, len(train_docs)):.2f} ({time.time() - t0:.0f}s)")
    return PAIR_CACHE[key]
_ = get_pairs(resolve(C))

### 2. Experiment runner

In [ ]:
import gc, traceback

def build_summ(c):
    arch, method = c["ARCH"], c["METHOD"]
    if SMOKE_TEST:
        cfg = dict(arch=arch, vocab_size=len(tokenizer), n_embd=32, n_layer=4, n_enc_layer=4, n_dec_layer=2, n_head=4,
                   ffn_hidden=64, rope_theta=1e4, dropout=0.0, max_positions=1344,
                   bos_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
        bb = ARCH_CLASSES[arch](cfg)
    else:
        cfg, bb = load_backbone(find_model_dir(arch), ARCH_CLASSES)
        bb.cfg.setdefault("max_positions", cfg.get("max_positions", 1024))
    if method == "kd":
        bb = keep_layers(arch, bb, c["KD_KEEP_LAYERS"] if not SMOKE_TEST else [0, 2])
    m = SummModel(arch, bb, n_prompt=c["N_PROMPT"] if method == "prompt" else 0,
                  n_prefix=c["N_PREFIX"] if method == "prefix" else 0)
    need = c["SRC_LEN"] + c["TGT_LEN"] + m.n_virtual + 1
    assert need <= m.max_len(), f"SRC_LEN + TGT_LEN + prompt = {need} exceeds the model's {m.max_len()} positions"
    info = apply_peft(m, method, c)
    return m.to(device), info

def load_teacher(c):
    tdir = os.path.join(WORK, c["KD_TEACHER_RUN"]) if c.get("KD_TEACHER_RUN") else c["KD_TEACHER_DIR"]
    if not (tdir and os.path.exists(os.path.join(tdir, "summ_model.pt"))) and c.get("KD_TEACHER_DIR"):
        tdir = c["KD_TEACHER_DIR"]
    assert tdir and os.path.exists(os.path.join(tdir, "summ_model.pt")), f"teacher weights not found in {tdir}"
    tc = json.load(open(os.path.join(tdir, "run_config.json")))
    t, tinfo = build_summ(tc)
    t.load_state_dict(torch.load(os.path.join(tdir, "summ_model.pt"), map_location="cpu"), strict=False); t.eval()
    for p in t.parameters(): p.requires_grad = False
    print(f"teacher: {tc['RUN_NAME']} ({tinfo['total_params'] / 1e6:.1f}M params)")
    return t, tc["RUN_NAME"]

def run_experiment(c):
    c = resolve(c)
    random.seed(c["SEED"]); np.random.seed(c["SEED"]); torch.manual_seed(c["SEED"])
    out = os.path.join(WORK, c["RUN_NAME"]); os.makedirs(out, exist_ok=True)
    pad = tokenizer.eos_token_id
    train_pairs, val_pairs, med_tgt = get_pairs(c)
    if c["MIN_NEW"] is None: c["MIN_NEW"] = int(0.6 * min(med_tgt, c["TGT_LEN"]))
    model, info = build_summ(c)
    gen_desc = ("per-document length control" if c["LENGTH_CONTROL"]
                else f"min {c['MIN_NEW']}, max {c['SEG_MAX_NEW']} tokens per segment")
    print(f"{c['ARCH']} + {c['METHOD']} (long_doc {c['LONG_DOC']}, context {c['CONTEXT_SELECT']}, lr {c['LR']}): trainable "
          f"{info['trainable_params'] / 1e6:.2f}M / {info['total_params'] / 1e6:.2f}M ({info['trainable_pct']:.2f}%) | "
          f"generation: {gen_desc}, up to {c['MAX_SEGMENTS']} segments")
    teacher, tname = load_teacher(c) if c["METHOD"] == "kd" else (None, None)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    result = train(model, train_pairs, val_pairs, c, pad, device, teacher=teacher)
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    del teacher; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

    t0 = time.time(); val_pred = summarise(model, val_gen_docs, tokenizer, c, device); val_s = time.time() - t0
    scores = evaluate_summaries(val_gen_docs, val_pred, c, device)
    print(f"\nVALIDATION {c['RUN_NAME']} ({len(val_gen_docs)} docs): BERTScore-F1 {scores['bertscore_f1']} | ROUGE-1 "
          f"{scores['rouge1']:.4f} | ROUGE-2 {scores['rouge2']:.4f} | ROUGE-L {scores['rougeL']:.4f} | words {scores['pred_words']:.0f} "
          f"(reference {scores['ref_words']:.0f}) | repeated trigrams {scores['rep3']:.3f} | novel bigrams {scores['novel2']:.3f}")
    print("length curve (first x% of each summary):", {k: round(v["rouge1"], 4) for k, v in scores["length_curve"].items()})
    ex = val_gen_docs[0]
    print(f"\nexample {ex['id']}\nREFERENCE: {ex['summary'][:700]}\n\nGENERATED: {val_pred[ex['id']][:700]}\n")

    t0 = time.time(); test_pred = summarise(model, test_docs, tokenizer, c, device); test_s = time.time() - t0
    pred = [{"id": d["id"], "summary": test_pred[d["id"]]} for d in test_docs]
    assert len(pred) == len(test_docs) and all(p["summary"].strip() for p in pred)
    json.dump(pred, open(os.path.join(out, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out, "prediction.json"), "prediction.json")
    json.dump([{"id": d["id"], "reference": d["summary"], "generated": val_pred[d["id"]]} for d in val_gen_docs],
              open(os.path.join(out, "val_predictions.json"), "w"), indent=1)
    json.dump({k: v for k, v in c.items() if isinstance(v, (int, float, str, bool, list, dict, type(None)))},
              open(os.path.join(out, "run_config.json"), "w"), indent=2)
    metrics = {"run": c["RUN_NAME"], "arch": c["ARCH"], "method": c["METHOD"], "long_doc": c["LONG_DOC"],
               "context_select": c["CONTEXT_SELECT"], "lr": c["LR"], "teacher": tname, **{f"val_{k}": v for k, v in scores.items()},
               **info, "best_val_loss": result["best_val_loss"], "train_time_s": result["train_time_s"],
               "epochs_run": len(result["history"]), "history": result["history"], "peak_gpu_GB": peak,
               "val_generation_s": val_s, "test_generation_s": test_s,
               "test_words": float(np.mean([len(p["summary"].split()) for p in pred]))}
    json.dump(metrics, open(os.path.join(out, "metrics.json"), "w"), indent=2)
    if c["METHOD"] == "full" or c["SAVE_WEIGHTS"] or any(e.get("KD_TEACHER_RUN") == c["RUN_NAME"] for e in EXPERIMENTS):
        torch.save(model.state_dict(), os.path.join(out, "summ_model.pt"))
    print(f"saved {out} | train {result['train_time_s'] / 60:.1f} min | generation val {val_s / 60:.1f} + test {test_s / 60:.1f} min "
          f"| peak GPU {peak:.2f} GB")
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return metrics

### 3. Run the experiments of this set

In [ ]:
run_start, summary = time.time(), []
offset = 0 if RUN_SET == "A" else len(SETS["A"])
for i, e in enumerate(EXPERIMENTS, 1):
    c = experiment_config(e)
    used = (time.time() - run_start) / 3600
    longest = max([m["train_time_s"] + m["val_generation_s"] + m["test_generation_s"] for m in summary if "train_time_s" in m] or [0]) / 3600
    if used + 1.2 * longest > TIME_BUDGET_HOURS:
        print(f"SKIP {c['RUN_NAME']}: time budget"); summary.append({"run": c["RUN_NAME"], "status": "skipped"}); continue
    print("\n" + "=" * 100 + f"\nEXPERIMENT {i}/{len(EXPERIMENTS)} (set {RUN_SET}): {c['RUN_NAME']}   ({used:.2f} h elapsed)\n" + "=" * 100)
    try:
        summary.append(dict(run_experiment(c), status="ok"))
    except Exception as ex:
        traceback.print_exc(); summary.append({"run": c["RUN_NAME"], "status": f"FAILED: {type(ex).__name__}: {ex}"})
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
print("\n" + "=" * 100 + "\nSUMMARY (validation = held-out part of train.json)")
print(f"{'run':40s} {'BERTSc':>7s} {'R1':>6s} {'R2':>6s} {'RL':>6s} {'words':>6s} {'rep3':>5s} {'vloss':>6s} {'trainable':>10s} {'min':>5s} {'GB':>5s}")
for m in summary:
    if m.get("status") != "ok": print(f"{m['run']:40s} {m['status']}"); continue
    bs = m["val_bertscore_f1"]
    print(f"{m['run']:40s} {('%.4f' % bs) if bs is not None else '   n/a':>7s} {m['val_rouge1']:6.4f} {m['val_rouge2']:6.4f} {m['val_rougeL']:6.4f} "
          f"{m['val_pred_words']:6.0f} {m['val_rep3']:5.3f} {m['best_val_loss']:6.3f} {m['trainable_params'] / 1e6:9.2f}M "
          f"{m['train_time_s'] / 60:5.1f} {m['peak_gpu_GB']:5.2f}")
json.dump(summary, open(os.path.join(WORK, f"phase6_summary_{RUN_SET}.json"), "w"), indent=2)
with zipfile.ZipFile(os.path.join(WORK, f"phase6_set{RUN_SET}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, f"phase6_summary_{RUN_SET}.json"), f"phase6_summary_{RUN_SET}.json")
    for i, m in enumerate(summary, 1):
        d = os.path.join(WORK, m["run"])
        if os.path.exists(os.path.join(d, "submission.zip")):
            z.write(os.path.join(d, "submission.zip"), f"codabench/{offset + i}_{m['run']}.zip")
        for f in ("metrics.json", "run_config.json", "val_predictions.json"):
            if os.path.exists(os.path.join(d, f)): z.write(os.path.join(d, f), f"{m['run']}/{f}")
print(f"Download phase6_set{RUN_SET}.zip; upload the files in its codabench/ folder to Codabench.")